In [ ]:
import pandas as pd
import torch
import copy
import numpy as np

df_annotations = pd.read_csv("annotations_data.csv")
df_openfda = pd.read_csv("openfda_data.csv")

drugs = ["Fluoxetine", "Escitalopram", "Sertraline", "Desvenlafaxine", "Duloxetine", 
         "Venlafaxine", "Amoxapine", "Amitriptyline", "Desipramine", "Trazodone", 
         "Nefazodone", "Mirtazapine", "Bupropion", "Selegiline", "Isocarboxazid", 
         "Phenelzine", "Esketamine", "Brexanolone", "Gepirone", "Zuranolone"]

#Deletes any extra spaces in the columns 
df_openfda.columns = df_openfda.columns.str.strip()
df_annotations.columns = df_annotations.columns.str.strip()

#Counts the total numbers of ALL adverse events for each drug
#(ex {"Fluoxetine: 45120, ..."}
fda_total_ae_count = df_openfda.groupby("Drug:")["Count"].sum().to_dict()

#Identifies the max total count of ADEs across all drugs, which acts as a scaling roof
max_count = max(fda_total_ae_count.values()) if fda_total_ae_count else 1

#Scales the raw report numbers into an unweighted 0-100 score, which represents
#the baseline toxicity for the chemical compound itself
min_count = min(fda_total_ae_count.values())
max_count = max(fda_total_ae_count.values())

drug_baseline_toxicity = {
    drug: ((count - min_count) / (max_count - min_count)) * 100 
    for drug, count in fda_total_ae_count.items()
}



#Stores the completed multimodal profile graphs 
multi_dataset = []

#Gets all the unique genes to build a fixed-size index for the 
#one-hot encoding 
unique_genes = list(df_annotations["Gene"].dropna().unique())

#Traverses through each row 
for i, row in df_annotations.iterrows():
    raw_row_drugs = str(row["Drug(s)"]).lower() 
    risk_score = float(row["Risk_Score"])
    current_gene = str(row["Gene"])

    for d_name in drugs:
        #Since some of the strings contain multiple drugs (drug1, drug2, etc.)
        if d_name.lower() in raw_row_drugs:
            if d_name in drug_graphs:
                #Gets the baseline toxicity from openFDA (0.0 if it's missing)
                raw_toxicity = drug_baseline_toxicity.get(d_name, 0.0)

                #Scales the toxicity rating based on the specific risk_score
                #(2) = 50% increase, (1) = 25% increase
                new_toxicity = raw_toxicity * (1.0 + (risk_score *0.25)) 
                #Toxicity must be smaller than a 100
                new_toxicity = min(new_toxicity, 100.0)

                #Creates a deep copy of the molecular graph to prevent global mutation bugs
                sample = copy.deepcopy(drug_graphs[d_name])

                #Generate an array of 0s and one 1, which represents which specific enzyme is active
                gene_vector = [1 if current_gene == g else 0 for g in unique_genes]
                #Appends the quantitative risk scale at index 0 
                full_genetic_vector = [risk_score] + gene_vector

                #Appends patient genetics as a float feature tensor attribute
                sample.genetic_attr = torch.tensor([full_genetic_vector], dtype=torch.float)
                #Appends the multimodal scaled risk score for regression training 
                sample.y = torch.tensor([[new_toxicity]], dtype=torch.float)

                #Saves the full profile into the final dataset 
                multi_dataset.append(sample)

print("Completed")
print(f"Length of the dataset is {len(multi_dataset)}")


